<a href="https://colab.research.google.com/github/kny1209/test2/blob/main/AI/DR-08079_%EA%B5%AC%EB%82%98%EC%98%81_AI%EA%B0%9C%EB%A1%A0_16%EC%B0%A8%EC%8B%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

문항 1. 기존 MLP(Multi-Layer Perceptron) 모델은 이미지 처리에 근본적인 한계가 있습니다. 강의 자료에서 언급된 MLP의 한계점 2가지를 서술하시오.

답안:  

1. 이미지를 1차원 벡터로 변환하는 과정에서 픽셀 간의 공간 정보(공간적/위치적 관계 소실)가 손실된다.

2. 모든 픽셀이 다음 레이어의 모든 뉴런과 연결(Fully Connected)되므로, 이미지 크기가 조금만 커져도 파라미터 수가 기하급수적으로 증가하여 과적합 위험이 매우 높아져 학습이 불가능해진다. (파라미터 폭발)



문항 2. CNN의 풀링(Pooling) 레이어는 크게 두 가지 주요 역할을 수행합니다. 이 두 가지 역할을 서술하시오.


답안:

1. 공간 해상도(차원 축소)를 Feature Map의 크기를 줄여 계산량을 크게 감소시킨다.

2. 위치 불변성(Translation Invariance)확보 : 미세한 위치 변화에도 동일한 특징을 추출하도록 도와, 객체의 정확한 위치보다 존재 여부에 집중하게 한다.

문항 3. 데이터 전처리 정의 CIFAR-10 데이터셋을 불러오기 전, 이미지를 텐서로 변환하고 정규화(Normalize)하는 transform 객체를 정의하는 코드를 완성하시오. (조건: ToTensor()를 사용하고, 평균(mean)과 표준편차(std)는 모두 (0.5, 0.5, 0.5)로 설정하시오.)

In [ ]:
import torch
from torchvision import transforms

# [문제] transform 정의 부분을 채우시오.
transform = transforms.Compose([
    # 1. 이미지를 PyTorch Tensor로 변환
    transforms.ToTensor(),

    # 2. 정규화 (mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    transforms.Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5])
    # transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))
])

문항 4. CNN 모델 정의 - Feature Extractor SimpleCNN 클래스 내부의 self.features에 두 번째 합성곱 블록(Conv Block 2)을 추가하는 코드를 완성하시오.

(조건:

nn.Conv2d: 입력 채널 32, 출력 채널 64, 커널 크기 3, 패딩 1

nn.ReLU: 활성화 함수

nn.MaxPool2d: 커널 크기 2, 스트라이드 2)

In [ ]:
import torch.nn as nn

class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()

        # Feature Extractor 정의
        self.features = nn.Sequential(
            # Conv Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # [문제] Conv Block 2 코드를 아래에 완성하시오.
            nn.Conv2d(32, 64, kernel_size = 3, padding =1),     # padding=1을 적용해 3*3 커널 사용 시 공간 유지되도록 설정
            nn.ReLu(),
            nn.MaxPool2d(2, 2)
        )
        # ... (Classifier 부분 생략)

문항 5. CNN 모델 정의 - Classifier self.features를 통과한 특징 맵의 크기는 [배치 크기, 64, 8, 8]입니다. 이 3차원 텐서를 1차원으로 펼치고(Flatten), nn.Linear 층에 연결하는 self.classifier 코드를 완성하시오.

(조건:

nn.Linear의 첫 번째 입력 크기(in_features)는 64 * 8 * 8 (즉, 4096)입니다.

nn.Linear의 출력 크기(out_features)는 256입니다.)

In [ ]:
# ... (self.features 이후)

        # Classifier 정의
        self.classifier = nn.Sequential(
            # [문제 1] 텐서를 1차원으로 평탄화 (Flatten)
            nn.Flatten(),

            # [문제 2] Fully Connected Layer (입력: 4096, 출력: 256)
            nn.Linear(4096, 256),

            nn.ReLU(),
            nn.Linear(256, 10)  # CIFAR-10 클래스 10개
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

문항 6. 모델 평가 함수 구현 모델 평가(evaluate) 함수에서, 학습 모드(Dropout, Batch Normalization 등)를 비활성화하고 기울기(gradient) 계산을 중단하도록 하는 코드를 [문제] 위치에 각각 작성하시오.

In [ ]:
def evaluate(model, loader):
    # [문제 1] 모델을 평가 모드(evaluation mode)로 설정
    model.eval()        # Dropout 레이어 비활성화, BN 학습 시 평균.분산 사용하도록 고정

    correct = 0
    total = 0

    # [문제 2] 기울기 계산을 중단하는 컨텍스트 매니저
    with torch.no_grad():           # 불필요한 메모리 사용 줄이고, 계산 속도 향상
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    return accuracy